In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from delta.tables import DeltaTable

In [0]:
# Define schema
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("firstName", StringType(), True),
    StructField("lastName", StringType(), True),
    StructField("age", IntegerType(), True),
])

# Initial data for the target table
data_target = [
    (1, "Alice", "Anders", 30),
    (2, "Bob",   "Bauer",  40),
    (3, "Chris", "Conrad", 50),
]

df_target = spark.createDataFrame(data_target, schema)

# Write as a managed Delta table (adjust catalog.schema if needed)
df_target.write.format("delta").mode("overwrite").saveAsTable("default.people_demo")

display(spark.table("default.people_demo"))

In [0]:
data_updates = [
    (2, "Bob",  "Bauer",  41),   # existing id, age changed
    (4, "Dora", "Diehl",  28),   # new id, should be inserted
]

df_updates = spark.createDataFrame(data_updates, schema)

# Optionally: create temp view, as docs often do
df_updates.createOrReplaceTempView("people_demo_updates")

display(df_updates)

In [0]:
delta_table = DeltaTable.forName(spark, "default.people_demo")

(delta_table.alias("t")
    .merge(
        df_updates.alias("s"),
        "t.id = s.id"          # join condition
    )
    .whenMatchedUpdateAll()    # update all columns when ids match
    .whenNotMatchedInsertAll() # insert new rows when id not found
    .execute()
)

In [0]:
display(spark.table("default.people_demo").orderBy("id"))

In [0]:
#Update only certain columns
data_updates2 = [
    (1, "Alice", "Albrecht", 30),  # change lastName only
]
df_updates2 = spark.createDataFrame(data_updates2, schema)

(delta_table.alias("t")
    .merge(df_updates2.alias("s"), "t.id = s.id")
    .whenMatchedUpdate(set={
        "lastName": "s.lastName"    # only update the lastName
    })
    .execute()
)

display(spark.table("default.people_demo").orderBy("id"))

In [0]:
#Insert only when a condition is met 
data_updates3 = [
    (5, "Eve", "Engel", 25),  # age < 30, should NOT be inserted
    (6, "Frank", "Faber", 35) # age >= 30, should be inserted
]
df_updates3 = spark.createDataFrame(data_updates3, schema)

(delta_table.alias("t")
    .merge(df_updates3.alias("s"), "t.id = s.id")
    .whenNotMatchedInsert(
        condition="s.age >= 30",
        values={
            "id":       "s.id",
            "firstName":"s.firstName",
            "lastName": "s.lastName",
            "age":      "s.age"
        }
    )
    .execute()
)

display(spark.table("default.people_demo").orderBy("id"))


In [0]:
delta_table.history().show(truncate=False)